In [ ]:
# import kagglehub
# path = kagglehub.dataset_download("puneet6060/intel-image-classification")
# print("Path to dataset files:", path)

# !pip install torchutils

import torch
import torchvision
import torch.nn as nn
from torchvision import datasets
from torchvision import transforms as T

from torchvision import io
import torchutils as tu
from torch.utils.data import random_split, DataLoader, Subset
import json
import numpy as np
import matplotlib.pyplot as plt

: 

In [ ]:
path = 'd:\\Sirius26\\nn_project\\Ivan\\'

train_path = path + 'seg_train/seg_train'
test_path = path + 'seg_test/seg_test'
config = {
    'batch_size': 128,
    'learning_rate': 0.0005,
    'epochs': 5,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
}

In [ ]:
train_transform = T.Compose([
        T.Resize((224, 224)),
        T.RandomHorizontalFlip(p=0.5),
        T.RandomRotation(degrees=15),
        T.ColorJitter(
            brightness=0.2,
            contrast=0.2,
            saturation=0.2,
            hue=0.1
        ),
        T.ToTensor(),
        T.Normalize((0.5, 0.5, 0.5), (0.25, 0.25, 0.25)),
        T.RandomErasing(p=0.5, scale=(0.02, 0.2))
    ])

valid_transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize((0.5, 0.5, 0.5), (0.25, 0.25, 0.25)),
])

train_dataset = torchvision.datasets.ImageFolder(root=train_path, transform=train_transform)

valid_dataset = torchvision.datasets.ImageFolder(root=test_path, transform=valid_transform)

idx2class = {v: k for k, v in train_dataset.class_to_idx.items()}

train_loader = DataLoader(train_dataset, shuffle=True, batch_size=config['batch_size'])
valid_loader = DataLoader(valid_dataset, shuffle=True, batch_size=config['batch_size'])

In [ ]:
idx2class

In [ ]:
from torchvision.models import convnext_base, ConvNeXt_Base_Weights
model = convnext_base(weights=ConvNeXt_Base_Weights.DEFAULT)
model.to(config['device'])
# for param in model.parameters():
#     param.requires_grad = False


model.classifier[2] = nn.Linear(1024, 6)

for param in model.classifier.parameters():
    param.requires_grad = True

model.to(config['device'])

optimizer = torch.optim.Adam(model.parameters(), config['learning_rate'])
criterion = nn.CrossEntropyLoss()

def plot_history(history, grid=True):
    fig, ax = plt.subplots(1,2, figsize=(14,5))

    ax[0].plot(history['train_losses'], label='train loss')
    ax[0].plot(history['valid_losses'], label='valid loss')
    ax[0].set_title(f'Loss on epoch {len(history["train_losses"])}')
    ax[0].grid(grid)
    ax[0].set_ylim((0, max(history['train_losses'] + history['valid_losses']) + .1))
    ax[0].legend()

    ax[1].plot(history['train_accs'], label='train acc')
    ax[1].plot(history['valid_accs'], label='valid acc')
    ax[1].set_title(f'Accuracy on epoch {len(history["train_losses"])}')
    ax[1].grid(grid)
    ax[1].set_ylim((0, 1))
    ax[1].legend()

    plt.show()

def fit(
        model: torch.nn.Module,
        n_epochs: int,
        optimizer: torch.optim.Optimizer,
        train_loader: DataLoader,
        valid_loader: DataLoader,
        history=None,
        save_best=True,
        model_name="best_model",
        start_epoch=1,
        ) -> tuple[list, ...]:

    history = history or {
        'train_accs': [],
        'train_losses': [],
        'valid_accs': [],
        'valid_losses': [],
        'best_val_acc': 0,
        'best_epoch': 0
    }

    best_val_acc = history.get('best_val_acc', 0)

    for epoch in range(start_epoch, n_epochs+start_epoch):
        model.train()
        batch_losses = []
        batch_accs = []

        for images, labels in train_loader:
            images = images.to(config['device'])
            labels = labels.to(config['device'])

            y_pred = model(images)
            loss = criterion(y_pred, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            batch_losses.append(loss.item())
            batch_accs.append((y_pred.argmax(dim=1) == labels).float().mean().item())

        train_loss = np.mean(batch_losses)
        train_acc  = np.mean(batch_accs)
        history['train_losses'].append(train_loss)
        history['train_accs'].append(train_acc)

        model.eval()
        batch_losses = []
        batch_accs = []
        with torch.no_grad():
            for samples, labels in valid_loader:
                samples = samples.to(config['device'])
                labels = labels.to(config['device'])
                y_pred = model(samples)
                loss = criterion(y_pred, labels)
                batch_losses.append(loss.item())
                batch_accs.append((y_pred.argmax(dim=1) == labels).float().mean().item())

        valid_loss = np.mean(batch_losses)
        valid_acc  = np.mean(batch_accs)
        history['valid_losses'].append(valid_loss)
        history['valid_accs'].append(valid_acc)

        # ========== АВТОМАТИЧЕСКОЕ СОХРАНЕНИЕ ЛУЧШЕЙ МОДЕЛИ ==========
        if save_best and valid_acc > best_val_acc:
            best_val_acc = valid_acc
            best_epoch = epoch
            history['best_val_acc'] = best_val_acc
            history['best_epoch'] = best_epoch

            # Сохраняем модель с указанием точности и эпохи
            save_path = f'{model_name}_{best_val_acc:.4f}_epoch{best_epoch}.pth'
            torch.save(model.state_dict(), save_path)
            print(f"💾 Лучшая модель сохранена: {best_val_acc:.4f}% (эпоха {best_epoch})")
            print(f"   Файл: {save_path}")

        print(
            f"Epoch {epoch}/{n_epochs}:\n"
            f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}\n"
            f"Val Loss: {valid_loss:.4f},   Val Acc: {valid_acc:.4f}\n"
        )


    if save_best and history['best_val_acc'] > 0:
        best_path = f'{model_name}_{history["best_val_acc"]:.4f}_epoch{history["best_epoch"]}.pth'
        model.load_state_dict(torch.load(best_path))
        print(f"\n🏆 Загружена лучшая модель: {history['best_val_acc']:.4f}% с эпохи {history['best_epoch']}")

    plot_history(history)
    return history

In [ ]:
# import torch
# torch.cuda.empty_cache()

In [ ]:
fit(model, config['epochs'], optimizer, train_loader, valid_loader)

In [ ]:
# checkpoint = torch.load('best_model_0.9305_epoch9.pth', map_location=config['device'])
# model.load_state_dict(checkpoint)
# print("Модель загружена из best_model_0.9305_epoch9.pth")

In [ ]:
# history = {
#     'train_accs': [0.9265],      # значение с эпохи 9 (Train Acc)
#     'train_losses': [0.2054],    # Train Loss
#     'valid_accs': [0.9305],      # Val Acc
#     'valid_losses': [0.1899],    # Val Loss
#     'best_val_acc': 0.9305,
#     'best_epoch': 9
# }

In [ ]:
# for param_group in optimizer.param_groups:
#     param_group['lr'] = 0.0001

# fit(model, 20, optimizer, train_loader, valid_loader, history=history, start_epoch=10)

In [ ]:
# def get_batch_from_loader(loader):
#     batch, labels = next(iter(loader))
#     return batch, labels

# model.eval()

# batch, labels = get_batch_from_loader(valid_loader)
# batch = batch.to(config["device"]).float()

# with torch.no_grad():
#     logits = model(batch)
#     preds = torch.argmax(logits, dim=1).cpu().numpy()

# fig, ax = plt.subplots(8, 8, figsize=(30, 24))
# ax = ax.flatten()

# for i in range(len(ax)):
#     img = batch.cpu()[i].permute(1, 2, 0)
#     img = img * 0.25 + 0.5
#     img = torch.clamp(img, 0, 1)
#     ax[i].imshow(img)

#     true_label = labels[i].item()
#     pred_label = preds[i]

#     if true_label == pred_label:
#         color = 'green'
#     else:
#         color = 'red'

#     title = f"Real: {idx2class[true_label]}\nPred: {idx2class[pred_label]}"
#     ax[i].set_title(title, fontsize=12, color=color)
#     ax[i].axis('off')

# plt.tight_layout()
# plt.subplots_adjust(wspace=0.001, hspace=0.3)
# plt.show()